# Phase 1A + 1C | Human Neural Signal Preprocessing and Public Dataset Screening
### DANDI 001132 | Andrews et al. human hippocampal HD-MEA recordings

**Goal 1 (Phase 1A):** Read only a few seconds from selected electrodes of real NWB recordings over the internet, calculate conservative **candidate extracellular event features** and generate reproducible plots.

**Goal 2 (Phase 1C):** Independently screen alternative public optogenetic datasets for **explicit intervention time series**. No author contact and no manual relabeling.

**What we WILL NOT claim:** These are not clinically annotated seizures, single-neuron spike trains, validated neuronal firing rates, or verified stimulation-response labels. Raw threshold detections may include noise and artifacts. The alternative mouse datasets must not be pooled with human data as if directly comparable.

**No Kaggle input is required by default.** Turn Kaggle **Internet ON** and use **CPU**. We do **not** download the entire 192.4 GB Dandiset. The first run reads a small bounded time-by-electrode block from a single NWB asset. The optional alternate-dataset scan uses NWB metadata, not full ephys arrays.

Reference sources:
- Human recordings: https://dandiarchive.org/dandiset/001132 (published version 0.241130.1903)
- DANDI streaming: https://docs.dandiarchive.org/user-guide-using/accessing-data/streaming/
- Mouse hippocampal optogenetics: https://dandiarchive.org/dandiset/000568
- Mouse thalamocortical optogenetic inhibition: https://dandiarchive.org/dandiset/000009
- Documented optogenetic timing (mouse behavioral experiments): https://docs.dandiarchive.org/example-notebooks/000971/lernerlab/seiler_2024/optogenetics_example_notebook/

## 1. Install one lightweight streaming dependency

In [1]:
%pip -q install remfile

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration (safe starter settings)

First run: **one NWB recording**, **1.5 seconds**, **12 adjacent channels**. This is for verifying the pipeline only, **not** for model training. If remote reads time out, reduce `SEGMENT_SECONDS` to 0.5 or `NUM_ELECTRODES` to 4. GPU is unnecessary.

For cross-patient feature exploration later, increase `MAX_HUMAN_FILES` to 2-3; importantly, never randomly split windows from one donor across train and test.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, time, traceback, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import h5py
import remfile
from scipy.signal import butter, sosfiltfilt, find_peaks
from IPython.display import display

OUT = Path('/kaggle/working/phase1_research_outputs') if Path('/kaggle/working').exists() else Path('phase1_research_outputs')
OUT.mkdir(parents=True, exist_ok=True)

DANDI_VERSION = '0.241130.1903'
API = 'https://api.dandiarchive.org/api'
# These assets come from our prior version-pinned Phase 0 manifest.
HUMAN_TARGETS = [
    dict(slice_id='3C', donor='C', rec='3C-2', asset_id='495aa69c-4ee8-4553-abfa-298a23394466', opsin='HcKCR1'),
    dict(slice_id='7D', donor='D', rec='7D-7', asset_id='226915c5-faf6-4a48-9cda-b25b4ef2b98b', opsin='HcKCR1'),
    dict(slice_id='10F',donor='F', rec='10F-1',asset_id='1f3c9771-6abe-4af9-aaac-e7d7606784fa', opsin='HcKCR1'),
]
MAX_HUMAN_FILES = 1
START_SECOND = 1.0
SEGMENT_SECONDS = 1.5
ELECTRODE_START = 0
NUM_ELECTRODES = 12
EVENT_THRESHOLD_SIGMAS = 5.0
MIN_PEAK_SEPARATION_MS = 1.0
FEATURE_BIN_SECONDS = 0.25
HIGH_PASS_HZ = 300.0
RUN_ALTERNATIVE_METADATA_SCREEN = True
# Alternative checks are bounded and may be slower than the human 1.5-s read.
MAX_ALTERNATIVE_ASSETS_TO_PROBE = 1  # per archive
REQUEST_TIMEOUT = 50

print('Output:', OUT.resolve())
print('Human recordings to test:', MAX_HUMAN_FILES)
print('First recording window:', SEGMENT_SECONDS, 's x', NUM_ELECTRODES, 'electrodes')
print('No model training. No optical event labels are assumed.')

Output: /kaggle/working/phase1_research_outputs
Human recordings to test: 1
First recording window: 1.5 s x 12 electrodes
No model training. No optical event labels are assumed.


## 3. Remote HDF5 helper

DANDI's `/assets/<asset-id>/download/` endpoint redirects to cloud object storage. `remfile` supports byte-range requests, and `h5py` reads only requested HDF5 blocks. Some HDF5 compression/chunking may cause a larger amount of data to transfer than the requested NumPy block; check network usage if you expand the settings.

In [3]:
http = requests.Session()
http.headers.update({'User-Agent':'optogenetics-phase1a-reproducible-audit/1.0'})

def get_cloud_url(asset_id):
    url = f'{API}/assets/{asset_id}/download/'
    r = http.get(url, stream=True, allow_redirects=True, timeout=REQUEST_TIMEOUT)
    try:
        r.raise_for_status()
        if r.url == url or r.url.rstrip('/').endswith('/download'):
            raise RuntimeError('DANDI download endpoint did not redirect to a downloadable object.')
        return r.url
    finally:
        r.close()

def open_remote_nwb(asset_id):
    remote_file = remfile.File(url=get_cloud_url(asset_id))
    try:
        h5 = h5py.File(remote_file, mode='r')
        return remote_file, h5
    except Exception:
        remote_file.close()
        raise

def close_remote_nwb(remote_file, h5):
    try: h5.close()
    finally: remote_file.close()

def text_value(dataset):
    try:
        x = dataset[()]
        if isinstance(x, bytes): return x.decode('utf-8', 'replace')
        return str(x)
    except Exception:
        return ''

def electrical_series(h):
    # No blind recursive reading of arrays. The actual DANDI 001132 path was
    # confirmed in the Phase 0B metadata scans.
    path = '/acquisition/ElectricalSeries'
    if path not in h or 'data' not in h[path]:
        raise KeyError(f'Expected ElectricalSeries not found: {path}')
    return h[path]

def extract_rate(series):
    if 'starting_time' not in series or 'rate' not in series['starting_time'].attrs:
        raise ValueError('Sampling rate is not explicitly available; cannot infer time axis.')
    rate = float(np.asarray(series['starting_time'].attrs['rate']).item())
    if not 1000 <= rate <= 100000:
        raise ValueError(f'Unexpected sample rate: {rate}')
    return rate

def data_scaling_metadata(series):
    out = {}
    for key in ('conversion','offset','unit','resolution'):
        if key in series:
            out[key] = text_value(series[key]) if isinstance(series[key], h5py.Dataset) and series[key].shape == () else 'non-scalar'
        elif key in series['data'].attrs:
            out[key] = str(series['data'].attrs[key])
        else:
            out[key] = 'not_recorded_in_series'
    out['IMPORTANT'] = 'If conversion and offset are absent or ambiguous, use relative digital units; do not label plots in microvolts.'
    return out

def optical_metadata_snapshot(h):
    stim = sorted(list(h['/stimulus/presentation'].keys())) if '/stimulus/presentation' in h else []
    templ = sorted(list(h['/stimulus/templates'].keys())) if '/stimulus/templates' in h else []
    intervals = sorted(list(h['/intervals'].keys())) if '/intervals' in h else []
    opto_named = [s for s in stim + templ + intervals if re.search('opto|light|laser|gpio|ttl|trigger|led|optic',s,re.I)]
    return dict(stimulus_presentation_children=stim, stimulus_template_children=templ,interval_tables=intervals,
                optical_named_objects=opto_named, verified_optical_stim_events=0,
                interpretation='Group names are not timestamped labels; verify actual records before using stimulation targets.')

## 4. Read **one bounded block** from the human NWB recording

This cell must never call `data[:]`: it takes only a short time slice and a small **contiguous** set of channels. Note that NWB `subject_id='3C-2'` describes a recording. Published donor **C** is the independent biological group.

In [4]:
human_raw_segments = {}
human_provenance = []
read_errors = []
for target in HUMAN_TARGETS[:MAX_HUMAN_FILES]:
    rf = h = None
    print(f"Opening {target['rec']} (donor {target['donor']})...",flush=True)
    try:
        rf, h = open_remote_nwb(target['asset_id'])
        s = electrical_series(h)
        data = s['data']
        rate = extract_rate(s)
        start_idx = int(round(START_SECOND * rate))
        stop_idx = min(data.shape[0],start_idx+int(round(SEGMENT_SECONDS * rate)))
        start_ch = ELECTRODE_START
        stop_ch = min(data.shape[1], start_ch + NUM_ELECTRODES)
        if start_idx >= stop_idx or start_ch >= stop_ch:
            raise ValueError('Requested time/channel window outside file')
        before = time.monotonic()
        raw = np.asarray(data[start_idx:stop_idx,start_ch:stop_ch],dtype=np.float64)
        wall_seconds = time.monotonic()-before
        if raw.ndim != 2 or raw.size == 0:
            raise ValueError('Empty data slice returned')
        r = dict(target, n_total_samples=int(data.shape[0]), n_total_channels=int(data.shape[1]),
                 sample_rate_hz=rate, data_dtype=str(data.dtype), h5_chunk_shape=str(data.chunks),
                 segment_start_s=start_idx/rate, segment_end_s=stop_idx/rate,
                 channels=f'{start_ch}:{stop_ch}', samples_read=int(raw.shape[0]),
                 electrodes_read=int(raw.shape[1]), read_wall_seconds=round(wall_seconds,2),
                 nwb_subject_id=text_value(h['/general/subject/subject_id']) if '/general/subject/subject_id' in h else '',
                 scaling=data_scaling_metadata(s), optical_snapshot=optical_metadata_snapshot(h),
                 is_stimulation_labeled=False,
                 signal_units='relative raw digital units unless scaling independently validated')
        human_raw_segments[target['rec']] = (raw, rate, r)
        human_provenance.append(r)
        print(f'  OK: {raw.shape}, {rate:g} Hz; remote block read {wall_seconds:.1f}s',flush=True)
    except Exception as exc:
        msg=dict(recording=target['rec'],error=repr(exc),traceback=traceback.format_exc()[-1800:])
        read_errors.append(msg)
        print(f'  ERROR: {repr(exc)}',flush=True)
    finally:
        if rf is not None and h is not None:
            close_remote_nwb(rf,h)
        elif rf is not None:
            rf.close()

(OUT/'human_recording_provenance.json').write_text(json.dumps(human_provenance,indent=2,default=str))
(OUT/'human_read_errors.json').write_text(json.dumps(read_errors,indent=2))
print(f'Opened {len(human_raw_segments)}/{min(MAX_HUMAN_FILES,len(HUMAN_TARGETS))} requested files.')
if len(human_raw_segments)==0:
    print('NO REAL FEATURES will be generated: inspect human_read_errors.json, turn on Internet, or reduce segment settings.')

Opening 3C-2 (donor C)...
  OK: (30000, 12), 20000 Hz; remote block read 0.7s
Opened 1/1 requested files.


## 5. A conservative candidate event detector (NOT spike sorting)

**Important:** This detects negative excursions on each **electrode**, not unique neurons. One neuron can appear on several electrodes, and threshold peaks may include noise, stimulation artifacts, or burst overlap. We use high-pass filtering, a median-absolute-deviation (MAD) threshold, and a refractory distance to create features for future quality-controlled analyses.

In [5]:
def extract_candidate_events(raw, fs, threshold_sigmas=5.0, high_pass_hz=300.0, min_separation_ms=1.0):
    """Return high-pass signals, per-electrode candidate-negative-peak indices and channel diagnostics.
    raw must be a small time x electrode array from a bounded NWB read.
    No biological spike sorting or event labeling is performed.
    """
    arr = np.asarray(raw,dtype=np.float64)
    if arr.ndim != 2 or arr.shape[0] < 100:
        raise ValueError('Expected 2D time x electrode signal with >=100 samples.')
    if np.any(~np.isfinite(arr)):
        raise ValueError('Non-finite signal values. Inspect recording before analysis.')
    if not (0 < high_pass_hz < fs/2):
        raise ValueError('Invalid high-pass cutoff for sampling rate.')
    centered=arr-np.median(arr,axis=0,keepdims=True)
    filt = butter(3, high_pass_hz, btype='highpass', fs=fs, output='sos')
    filtered = sosfiltfilt(filt,centered,axis=0)
    # Discard 50ms at each edge to reduce filter-boundary candidate artifacts.
    edge = min(int(0.05*fs),int(0.10*len(arr)))
    peaks_by_ch=[]; diagnostics=[]
    max_peaks_guard = int(np.ceil(len(arr)/max(1,int(fs*min_separation_ms/1000))))
    for ci in range(arr.shape[1]):
        wave=filtered[:,ci]
        core=wave[edge:len(wave)-edge] if len(wave)>2*edge else wave
        median=np.median(core)
        sigma=1.4826*np.median(np.abs(core-median))
        valid = np.isfinite(sigma) and sigma>0
        raw_constant = (np.max(arr[:,ci]) == np.min(arr[:,ci]))
        if valid and not raw_constant:
            idx,properties=find_peaks(-wave,height=threshold_sigmas*sigma,
                                     distance=max(1,int(round(min_separation_ms*fs/1000))))
            idx=idx[(idx>=edge)&(idx<(len(wave)-edge))]
            if len(idx)>max_peaks_guard: raise ValueError('Unexpected peak-count overflow')
        else:
            idx=np.asarray([],dtype=np.int64)
        peaks_by_ch.append(idx)
        diagnostics.append(dict(electrode_local_index=ci,noise_sigma_relative_units=float(sigma),
                                highpass_rms_relative_units=float(np.sqrt(np.mean(core**2))),
                                candidate_peak_count=int(len(idx)), constant_raw_channel=bool(raw_constant),
                                threshold_relative_units=float(threshold_sigmas*sigma) if valid else None))
    return filtered,peaks_by_ch, pd.DataFrame(diagnostics)

def binned_candidate_features(raw, fs, peaks_by_ch, filtered, bin_seconds=0.25):
    n=raw.shape[0]
    bin_size=max(1,int(round(bin_seconds*fs)))
    feats=[]
    for start in range(0,n,bin_size):
        end=min(n,start+bin_size)
        duration=(end-start)/fs
        counts=np.array([np.sum((idx>=start)&(idx<end)) for idx in peaks_by_ch],dtype=int)
        rms=np.sqrt(np.mean(filtered[start:end,:]**2,axis=0))
        feats.append(dict(window_start_s=start/fs,window_end_s=end/fs,duration_s=duration,
                          summed_candidate_peaks=int(counts.sum()),
                          candidate_peak_rate_all_electrodes_hz=float(counts.sum()/duration),
                          mean_candidate_peak_rate_per_electrode_hz=float(counts.mean()/duration),
                          active_electrode_fraction=float(np.mean(counts>0)),
                          median_highpass_rms_relative_units=float(np.median(rms)),
                          individual_electrode_peak_counts=counts))
    return feats

## 6. Extract reproducible features and save real plots

All saved files are tagged with source recording and donor. Plots are generated from data returned by the DANDI server; nothing is AI-generated. They are **exploratory recording plots**, not scientific evidence of epileptiform events or light effects.

In [6]:
all_feature_rows=[]
all_channel_rows=[]
feature_errors=[]
for rec,(raw,fs,meta) in human_raw_segments.items():
    try:
        filt,peaks,ch_df=extract_candidate_events(raw,fs,EVENT_THRESHOLD_SIGMAS,HIGH_PASS_HZ,MIN_PEAK_SEPARATION_MS)
        rows=binned_candidate_features(raw,fs,peaks,filt,FEATURE_BIN_SECONDS)
        base = dict(dandiset='001132',dandiset_version=DANDI_VERSION, recording=rec,
                    slice_id=meta['slice_id'],donor_id=meta['donor'],
                    validated_light_timing=False,clinical_seizure_labels=False,
                    is_spike_sorted=False,high_pass_hz=HIGH_PASS_HZ,
                    threshold_mad_sigmas=EVENT_THRESHOLD_SIGMAS)
        for row in rows:
            feature={k:v for k,v in row.items() if k!='individual_electrode_peak_counts'}
            feature['time_start_nwb_s']=meta['segment_start_s']+feature.pop('window_start_s')
            feature['time_end_nwb_s']=meta['segment_start_s']+feature.pop('window_end_s')
            all_feature_rows.append(dict(base,**feature))
        ch_df['recording']=rec;ch_df['donor_id']=meta['donor'];ch_df['slice_id']=meta['slice_id']
        ch_df['electrode_dataset_column']=np.arange(ELECTRODE_START,ELECTRODE_START+len(ch_df))
        all_channel_rows.append(ch_df)
        # Preview: independent electrode signals shifted for readability.
        n_show=min(5,filt.shape[1]); tt=np.arange(len(filt))/fs+meta['segment_start_s']
        fscale=np.median([np.std(filt[:,j]) for j in range(n_show)])
        if not np.isfinite(fscale) or fscale<1e-9: fscale=1.0
        fig,ax=plt.subplots(figsize=(10,5))
        for ch in range(n_show):
            offset=(n_show-1-ch)*8*fscale
            ax.plot(tt,filt[:,ch]+offset,lw=0.55,label=f'electrode col {ELECTRODE_START+ch}')
            ax.scatter(tt[peaks[ch]],filt[peaks[ch],ch]+offset,s=12,marker='x')
        ax.set_title(f'{rec}: high-pass electrode traces + candidate peaks (unlabeled)')
        ax.set_xlabel('Time in recording (s)');ax.set_ylabel('Relative digital signal + vertical offset')
        ax.legend(loc='upper right',fontsize=8); fig.tight_layout()
        fig.savefig(OUT/f'{rec}_candidate_events.png',dpi=160);plt.close(fig)
        # Network candidate event rate across selected electrodes.
        x=[r['window_start_s'] + meta['segment_start_s'] for r in rows]
        y=[r['candidate_peak_rate_all_electrodes_hz'] for r in rows]
        fig,ax=plt.subplots(figsize=(8,3.5))
        ax.plot(x,y,marker='o');ax.set_xlabel('Time in recording (s)')
        ax.set_ylabel('Candidate peaks/s (summed across selected electrodes)')
        ax.set_title(f'{rec}: exploratory electrode-event activity, NOT seizures')
        fig.tight_layout();fig.savefig(OUT/f'{rec}_candidate_network_rate.png',dpi=160);plt.close(fig)
        print(f'{rec}: {len(rows)} feature windows and {len(ch_df)} electrode summaries')
    except Exception as exc:
        feature_errors.append(dict(recording=rec,error=repr(exc),traceback=traceback.format_exc()[-1800:]))
        print('Feature extraction error:',rec,repr(exc))

features=pd.DataFrame(all_feature_rows)
channels=pd.concat(all_channel_rows,ignore_index=True) if all_channel_rows else pd.DataFrame()
features.to_csv(OUT/'human_window_features_UNLABELED.csv',index=False)
channels.to_csv(OUT/'human_channel_quality_and_candidate_peaks.csv',index=False)
(OUT/'human_feature_errors.json').write_text(json.dumps(feature_errors,indent=2))
if features.empty:
    print('No empirical features created. Check saved errors; do NOT train on dummy data.')
else:
    display(features.head(12))
    print('UNLABELED exploratory windows:',len(features),' unique donors:',features.donor_id.nunique())
    print('These rows are NOT statistically independent. Group by donor for any later evaluation.')

3C-2: 6 feature windows and 12 electrode summaries


,dandiset,dandiset_version,recording,slice_id,donor_id,validated_light_timing,clinical_seizure_labels,is_spike_sorted,high_pass_hz,threshold_mad_sigmas,duration_s,summed_candidate_peaks,candidate_peak_rate_all_electrodes_hz,mean_candidate_peak_rate_per_electrode_hz,active_electrode_fraction,median_highpass_rms_relative_units,time_start_nwb_s,time_end_nwb_s
0,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.854524,1.00,1.25
1,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.863044,1.25,1.50
2,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.848606,1.50,1.75
3,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.842647,1.75,2.00
4,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.815006,2.00,2.25
5,001132,0.241130.1903,3C-2,3C,C,False,False,False,300.0,5.0,0.25,0,0.0,0.0,0.0,4.911687,2.25,2.50


UNLABELED exploratory windows: 6  unique donors: 1
These rows are NOT statistically independent. Group by donor for any later evaluation.


## 7. Optional dataset audit: is any real stimulation timing accessible?

The DANDI 001132 file's `/stimulus` groups may be present but empty. Record this explicitly. No guessing from raw voltage traces or authors' example logs. If the human NWB contains no aligned light events, our Phase 1A features remain **unlabeled**.

In [7]:
optical_rows=[]
for meta in human_provenance:
    snap=meta.get('optical_snapshot',{})
    optical_rows.append(dict(recording=meta['rec'], donor_id=meta['donor'],
      stimulus_presentation_count=len(snap.get('stimulus_presentation_children',[])),
      stimulus_presentation_names='|'.join(snap.get('stimulus_presentation_children',[])),
      interval_table_count=len(snap.get('interval_tables',[])),
      optical_named_objects='|'.join(snap.get('optical_named_objects',[])),
      n_verified_optical_trials=0,
      evidence_label='NWB metadata only; no verified intervention events'))
optical_df=pd.DataFrame(optical_rows)
optical_df.to_csv(OUT/'human_optical_label_audit.csv',index=False)
display(optical_df)

,recording,donor_id,stimulus_presentation_count,stimulus_presentation_names,interval_table_count,optical_named_objects,n_verified_optical_trials,evidence_label
0,3C-2,C,0,,0,,0,NWB metadata only; no verified intervention ev...


## 8. Phase 1C: screen alternative datasets (metadata-only)

This is **research triage**, not proof of transplantable data. We screen:

| Dandiset | Biological context | Potential use | Major limitation |
|---|---|---|---|
| **000568** | Mouse hippocampus + electrophysiology + optogenetic pulses | Best candidate for optical response dynamics | Mouse, not human epilepsy; 904.8 GB archive |
| **000009** | Mouse thalamo-cortical spiking + optogenetic inhibition | Timing and controlled neural-response methods | Different brain region and task |
| **000971** | Mouse dorsomedial-striatum optogenetics + behavior | **Known explicit OptogeneticSeries ON/OFF timings**; test event parser | Lacks matching extracellular neural recordings |

These are **not validated drop-in training datasets**. We must inspect NWB time series per recording. Our code reads only small metadata/scalars and up to 20 stimulus values when visible; no large electrical arrays.

In [8]:
ALTERNATIVE_SOURCES = [
    dict(dandiset='000568',version='0.230705.1633',species='mouse',role='mouse_hippocampus_optogenetic_neural_response',
         nwb_path=None, evidence_url='https://dandiarchive.org/dandiset/000568'),
    dict(dandiset='000009',version='0.220126.1903',species='mouse',role='mouse_thalamocortical_inhibition',
         nwb_path=None, evidence_url='https://dandiarchive.org/dandiset/000009'),
    dict(dandiset='000971',version='0.260213.1851',species='mouse',role='explicit_opto_times_behavior_only',
         nwb_path='sub-119-416/sub-119-416_ses-Opto-DMS-Excitatory-ChR2-2020-10-20T13-00-57_ogen.nwb',
         evidence_url='https://docs.dandiarchive.org/example-notebooks/000971/lernerlab/seiler_2024/optogenetics_example_notebook/'),
]

def get_assets_in_dandiset(ds_id, version, path_filter=None, max_pages=3):
    url=f'{API}/dandisets/{ds_id}/versions/{version}/assets/'
    query={'page_size':100}
    if path_filter: query['path']=path_filter
    collected=[]
    for page in range(max_pages):
        r=http.get(url,params=query if page==0 else None, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        p=r.json();collected.extend(p.get('results',[]))
        url=p.get('next'); query=None
        if not url: break
    return collected

def _series_metadata_only(h):
    # Search only the explicit stimuli group; its series may have data/timestamps arrays.
    names=[]
    if '/stimulus/presentation' in h:
        g=h['/stimulus/presentation']
        names=list(g.keys())
    items=[]
    for name in names[:25]:
        g=h['/stimulus/presentation'][name]
        if not isinstance(g,h5py.Group): continue
        desc=text_value(g['description'])[:180] if 'description' in g else ''
        d=g['data'] if 'data' in g and isinstance(g['data'],h5py.Dataset) else None
        ts=g['timestamps'] if 'timestamps' in g and isinstance(g['timestamps'],h5py.Dataset) else None
        # Read an ultra-short preview ONLY for small 1D time series.
        preview=[]
        if d is not None and ts is not None and d.ndim==1 and ts.ndim==1:
            k=min(20,len(d),len(ts))
            preview=[{'time_s':float(t),'value':float(v)} for t,v in zip(ts[:k],d[:k])]
        items.append(dict(name=name,description=desc,
                          data_shape=str(d.shape) if d is not None else '',
                          timestamps_shape=str(ts.shape) if ts is not None else '',
                          sample_preview=preview,
                          explicit_opto_name=bool(re.search('opto|light|laser|photo',name,re.I))))
    return items

def probe_alternative(ds,asset):
    report=dict(dandiset=ds['dandiset'],version=ds['version'],role=ds['role'],
                asset_path=asset.get('path',''),asset_id=asset['asset_id'],status='failed',
                has_electricalseries=False,has_units=False,stimulation_groups=[],
                explicit_series=[],error='')
    rf=h=None
    try:
        rf,h=open_remote_nwb(asset['asset_id'])
        report['has_electricalseries']='/acquisition/ElectricalSeries/data' in h
        report['has_units']='/units' in h
        report['stimulation_groups']=list(h['/stimulus/presentation'].keys()) if '/stimulus/presentation' in h else []
        report['explicit_series']=_series_metadata_only(h)
        report['status']='metadata_opened'
        report['label_readiness']='candidate_only_require_event_values_and_neural_alignment'
    except Exception as exc:
        report['error']=repr(exc)[:500]
    finally:
        if rf is not None and h is not None: close_remote_nwb(rf,h)
        elif rf is not None:rf.close()
    return report

alternative_manifest_summary=[]
alternative_probes=[]
if RUN_ALTERNATIVE_METADATA_SCREEN:
    for ds in ALTERNATIVE_SOURCES:
        try:
            assets=get_assets_in_dandiset(ds['dandiset'],ds['version'],ds['nwb_path'],max_pages=3)
            if ds['nwb_path']:
                assets=[a for a in assets if a.get('path')==ds['nwb_path']]
            nwb_assets=[a for a in assets if a.get('path','').lower().endswith('.nwb')]
            nwb_assets.sort(key=lambda a: a.get('size',10**99))
            preferred=[a for a in nwb_assets if any(k in a['path'].lower() for k in ('ecephys','_ephys','electrophys'))]
            targets=(preferred or nwb_assets)[:MAX_ALTERNATIVE_ASSETS_TO_PROBE]
            alternative_manifest_summary.append(dict(dandiset=ds['dandiset'],version=ds['version'],
                 assets_found_in_bounded_listing=len(assets),nwb_assets_found_in_bounded_listing=len(nwb_assets),
                 selected_probe_count=len(targets),note='bounded listing; NOT a dataset-wide count'))
            print('Screening',ds['dandiset'], 'candidate NWB files:',len(nwb_assets),'selected:',len(targets),flush=True)
            for asset in targets:
                report=probe_alternative(ds,asset)
                alternative_probes.append(report)
                print('  ',report['status'],asset['path'][-75:], 'series',report['stimulation_groups'],flush=True)
        except Exception as exc:
            alternative_probes.append(dict(dandiset=ds['dandiset'],status='manifest_error',error=repr(exc)))
            print('  Dataset scan error:',ds['dandiset'],repr(exc))
else:
    print('Alternative scan disabled. Set RUN_ALTERNATIVE_METADATA_SCREEN=True to inspect NWB metadata.')

pd.DataFrame(alternative_manifest_summary).to_csv(OUT/'alternative_dandi_manifest_screen.csv',index=False)
flat=[]
for r in alternative_probes:
    q={k:v for k,v in r.items() if k not in ('explicit_series','stimulation_groups')}
    q['stimulus_names']='|'.join(r.get('stimulation_groups',[]))
    q['stimulus_series_count']=len(r.get('explicit_series',[]))
    q['stimulation_timestamps_visible']=any(bool(s.get('sample_preview')) for s in r.get('explicit_series',[]))
    flat.append(q)
pd.DataFrame(flat).to_csv(OUT/'alternative_nwb_label_screen.csv',index=False)
(OUT/'alternative_nwb_metadata_details.json').write_text(json.dumps(alternative_probes,indent=2,default=str))
if flat:
    display(pd.DataFrame(flat)[['dandiset','status','has_electricalseries','has_units','stimulus_names','stimulation_timestamps_visible','error']])

Screening 000568 candidate NWB files: 56 selected: 1
   metadata_opened 2/sub-fCamk2_ses-fCamk2-201014-sess3-no-raw-data_behavior+ecephys+image.nwb series []
Screening 000009 candidate NWB files: 173 selected: 1
   metadata_opened sub-anm00260490/sub-anm00260490_ses-20170627T093718_ecephys+ogen.nwb series []
Screening 000971 candidate NWB files: 1 selected: 1
   metadata_opened 9-416/sub-119-416_ses-Opto-DMS-Excitatory-ChR2-2020-10-20T13-00-57_ogen.nwb series ['OptogeneticSeries']


,dandiset,status,has_electricalseries,has_units,stimulus_names,stimulation_timestamps_visible,error
0,000568,metadata_opened,False,True,,False,
1,000009,metadata_opened,False,True,,False,
2,000971,metadata_opened,False,False,OptogeneticSeries,True,


## 9. Scientific decision report and next steps

Choose a trajectory based on **real measured evidence**:

- Human raw traces available + no optical timestamps: **Phase 1A is feasible**, but train only *non-interventional* baseline models after a rigorous cross-donor pipeline. Do not label light or seizure outcomes.
- Alternative dataset has both neural recordings and explicit timed `OptogeneticSeries`: next audit precise event alignment and repeated perturbation conditions **within that dataset**. It may inform general modeling but not prove human epilepsy control.
- Explicit timestamps only + no electrical data (e.g., some behavioral datasets): good for understanding event schema, **not** a stimulation-response digital twin.
- Do not use pseudo-labels from unknown stimulation times, or mix windows from one biological donor across training/test.

In [9]:
summary={
 'utc_timestamp':datetime.now(timezone.utc).isoformat(),
 'human_dandiset':'001132','version':DANDI_VERSION,
 'human_nwb_requested':min(MAX_HUMAN_FILES,len(HUMAN_TARGETS)),
 'human_nwb_opened':len(human_raw_segments),
 'human_features_created':len(features),
 'human_distinct_donors_in_current_run':int(features.donor_id.nunique()) if not features.empty else 0,
 'human_verified_optical_events':0,
 'human_label_status':'NOT LABELED: no external stimulation logs or GPIO timestamps aligned',
 'alternative_nwb_files_probe_attempted':sum(('asset_id' in p) for p in alternative_probes),
 'alternative_probe_metadata_opened':sum(p.get('status')=='metadata_opened' for p in alternative_probes),
 'next_step_if_human_features_ok':'Run longer multi-slice quality assessment; split by donor; evaluate whether labels can be recovered independently',
 'next_step_if_alternative_event_series_ok':'Inspect all relevant stimulus changes, repeat trials and neural alignment in a selected alternative recording',
 'prohibited_claims':'No clinical seizure detection, no true neural firing rates from simple threshold counts, and no demonstrated optogenetic control.'
}
(OUT/'phase1_feasibility_decision.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
with zipfile.ZipFile(OUT.parent/'phase1_dandi_outputs.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file(): z.write(f,arcname=f.name)
print('\nZIP TO DOWNLOAD:',str(OUT.parent/'phase1_dandi_outputs.zip'))

{
  "utc_timestamp": "2026-10-09T21:57:03.409824+00:00",
  "human_dandiset": "001132",
  "version": "0.241130.1903",
  "human_nwb_requested": 1,
  "human_nwb_opened": 1,
  "human_features_created": 6,
  "human_distinct_donors_in_current_run": 1,
  "human_verified_optical_events": 0,
  "human_label_status": "NOT LABELED: no external stimulation logs or GPIO timestamps aligned",
  "alternative_nwb_files_probe_attempted": 3,
  "alternative_probe_metadata_opened": 3,
  "next_step_if_human_features_ok": "Run longer multi-slice quality assessment; split by donor; evaluate whether labels can be recovered independently",
  "next_step_if_alternative_event_series_ok": "Inspect all relevant stimulus changes, repeat trials and neural alignment in a selected alternative recording",
  "prohibited_claims": "No clinical seizure detection, no true neural firing rates from simple threshold counts, and no demonstrated optogenetic control."
}

ZIP TO DOWNLOAD: /kaggle/working/phase1_dandi_outputs.zip


## 10. What to send back to the research assistant

Download **`phase1_dandi_outputs.zip`** from `/kaggle/working/` and upload it to this chat. If a cell fails, upload the ZIP anyway (it contains saved errors where possible), and include the failing cell screenshot.

**Important:** On the starter run, the number of donors is one. That verifies code, not generalization. The follow-up experiment must include multiple donors and stricter quality controls before any Q1-oriented prediction claims.